# 08 · Segmentação: o contorno do coração

Aulas de **segmentação**: "tesoura em vez de moldura" (contorno real, não caixa rígida), "Dice: acordo
de área, não de caixa", "o pixel da borda pesa mais". Até aqui o coração foi medido por larguras
(notebook 04) e apontado por mapas de calor (07). Agora ele é **recortado**, e o recorte é comparado com
um **gabarito marcado à mão** pela equipe, na página de marcação, em 30 raio-X do teste.

1. **os contornos marcados** e o que eles mostram sobre quem marcou;
2. **três métodos automáticos**: caixa fixa, limiar e morfologia, bordas do ICT;
3. **Dice e IoU** contra o gabarito, com intervalo;
4. **onde erram**: área a mais ou a menos;
5. **o ICT do contorno à mão** contra o ICT automático;
6. **o Grad-CAM dentro do coração de verdade**, não da caixa;
7. **concordância entre anotadores**, quando houver mais de um.

Os parâmetros dos métodos automáticos foram fixados **antes** de comparar com o gabarito.

In [ ]:
import sys
from pathlib import Path

# Permite importar `src` mesmo fora do ambiente do uv (ex.: Colab com o repositório clonado).
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr

from src import config
from src.data.dados import baixar_base, carregar_imagens, dividir, listar_imagens
from src.model import avaliacao, cnn, gradcam
from src.utils.io import configurar_estilo, salvar_figura, salvar_tabela
from src.visao import ict, segmentacao
from src.visao.metricas import dice, iou

configurar_estilo()
SECAO = "08-segmentacao"
PARAMS_ICT = config.PARAMS_ICT  # escolhidos no notebook 04

baixar_base()
catalogo = dividir(listar_imagens())
selecao = pd.read_csv(config.DATA_PROCESSED / "marcacao_ids.csv", sep=";")
marcacoes = segmentacao.carregar_marcacoes()
anotadores = sorted(marcacoes["anotador"].unique())
NOMES = {a: f"anotador {k + 1}" for k, a in enumerate(anotadores)}
print(f"{len(marcacoes)} contornos de {len(anotadores)} anotador(es)")
print(marcacoes.groupby("anotador").size().rename(index=NOMES))

por_id = catalogo.set_index("id")
imagens = {i: carregar_imagens([por_id.loc[i, "caminho"]])[0] for i in selecao["id"]}
classe = {i: int(por_id.loc[i, "classe"]) for i in selecao["id"]}

# Caixa fixa do coração: a mesma dos notebooks 06 e 07, tirada das bordas do ICT no treino.
treino = catalogo[catalogo["conjunto"] == "treino"]
medidas_treino = [ict.medir_ict(im, **PARAMS_ICT) for im in carregar_imagens(treino["caminho"])]
CAIXA, _ = ict.caixas_de_referencia(medidas_treino)
print("Caixa fixa do coração:", CAIXA)

def contorno(ax, pontos, cor, **kw):
    p = np.asarray(pontos)
    ax.plot(*np.vstack([p, p[:1]]).T, color=cor, lw=kw.pop("lw", 1.6), **kw)

## 1. Os contornos marcados

Cada contorno é uma lista de pontos clicados sobre o raio-X, guardada pela página de marcação e
exportada para `data/processed/marcacoes/`. A ordem dos exames é a ordem em que foram apresentados.

In [ ]:
primeiro = anotadores[0]
dele = marcacoes[marcacoes["anotador"] == primeiro].set_index("id")["pontos"]
fig, eixos = plt.subplots(5, 6, figsize=(14, 12.5))
for ax, (_, r) in zip(eixos.ravel(), selecao.iterrows()):
    ax.imshow(imagens[r["id"]], cmap="gray")
    if r["id"] in dele.index:
        contorno(ax, dele[r["id"]], config.CORAL)
    ax.set_title(f"{r['ordem']} · {config.ROTULOS[classe[r['id']]]}" + ("" if r["id"] in dele.index else " · sem contorno"),
                 fontsize=8)
    ax.axis("off")
fig.suptitle(f"Contornos do {NOMES[primeiro]}", fontweight="bold", color=config.AZUL_PROFUNDO)
salvar_figura(fig, SECAO, "contornos-marcados")
plt.show()

area = selecao.assign(area=selecao["id"].map(lambda i: segmentacao.poligono_para_mascara(dele[i]).mean() if i in dele.index else np.nan),
                      pontos=selecao["id"].map(lambda i: len(dele[i]) if i in dele.index else 0))
salvar_tabela(area, SECAO, "area-por-ordem", indice=False)
print("Área do contorno (% da imagem), mediana: exames 1–4", f"{area['area'].iloc[:4].median():.1%}",
      "| exames 5–30", f"{area['area'].iloc[4:].median():.1%}")

## 2. Três métodos automáticos

Do mais simples ao mais informado:

* **caixa fixa**: a moldura dos notebooks 06 e 07, igual para todos;
* **limiar e morfologia**: Otsu calculado só na região central, abertura 7×7 para soltar o coração da
  coluna, o componente claro que contém o centro, fechamento 9×9 e preenchimento de buracos;
* **bordas do ICT**: em cada linha que o perfil do notebook 04 mediu, da borda direita à esquerda do
  coração.

In [ ]:
def mascaras_automaticas(imagem):
    medida = ict.medir_ict(imagem, **PARAMS_ICT)
    return {
        "caixa fixa": segmentacao.caixa_para_mascara(CAIXA),
        "limiar e morfologia": segmentacao.mascara_por_limiar(imagem),
        "bordas do ICT": segmentacao.mascara_pelo_ict(medida),
    }, medida

automaticas, medidas = {}, {}
for i in selecao["id"]:
    automaticas[i], medidas[i] = mascaras_automaticas(imagens[i])

exemplos = [i for i in selecao["id"] if i in dele.index][4:8]
CORES = {"caixa fixa": config.AMARELO, "limiar e morfologia": config.VERDE, "bordas do ICT": config.AZUL_CLARO}
fig, eixos = plt.subplots(len(exemplos), 4, figsize=(12, 3.1 * len(exemplos)))
for linha, i in enumerate(exemplos):
    gab = segmentacao.poligono_para_mascara(dele[i])
    eixos[linha, 0].imshow(imagens[i], cmap="gray")
    contorno(eixos[linha, 0], dele[i], config.CORAL)
    eixos[linha, 0].set_title(f"{i} · marcado à mão", fontsize=8)
    for col, (nome, m) in enumerate(automaticas[i].items(), start=1):
        ax = eixos[linha, col]
        ax.imshow(imagens[i], cmap="gray")
        ax.imshow(np.ma.masked_where(~m, m), cmap="winter", alpha=0.45)
        contorno(ax, dele[i], config.CORAL, lw=1.2)
        ax.set_title(f"{nome} · Dice {dice(gab, m):.2f}", fontsize=8)
    for ax in eixos[linha]:
        ax.axis("off")
salvar_figura(fig, SECAO, "metodos-exemplos")
plt.show()

## 3. Dice e IoU contra o gabarito

**Dice** = 2·|A∩B| / (|A|+|B|) e **IoU** = |A∩B| / |A∪B|, de 0 (nada em comum) a 1 (idênticas).
Intervalo de 95 % por bootstrap dos exames. Também a **cobertura** (quanto do gabarito o método pegou)
e a **precisão** (quanto do método está dentro do gabarito): Dice baixo por excesso e por falta são
erros diferentes.

In [ ]:
linhas = []
for _, r in marcacoes.iterrows():
    gab = segmentacao.poligono_para_mascara(r["pontos"])
    for nome, m in automaticas[r["id"]].items():
        inter = (gab & m).sum()
        linhas.append({"anotador": NOMES[r["anotador"]], "id": r["id"], "classe": classe[r["id"]], "metodo": nome,
                       "dice": dice(gab, m), "iou": iou(gab, m),
                       "cobertura": inter / max(gab.sum(), 1), "precisao": inter / max(m.sum(), 1),
                       "area_metodo": m.mean(), "area_gabarito": gab.mean()})
comparacao = pd.DataFrame(linhas)
salvar_tabela(comparacao, SECAO, "dice-por-exame", indice=False)

rng = np.random.default_rng(config.SEMENTE)
def ic_media(v, n=2000):
    v = np.asarray(v)
    boot = [rng.choice(v, len(v)).mean() for _ in range(n)]
    return np.percentile(boot, [2.5, 97.5])

resumo = []
for nome, g in comparacao.groupby("metodo", sort=False):
    baixo, alto = ic_media(g["dice"])
    resumo.append({"metodo": nome, "n": len(g), "dice_medio": g["dice"].mean(), "ic_baixo": baixo, "ic_alto": alto,
                   "dice_mediano": g["dice"].median(), "iou_medio": g["iou"].mean(),
                   "cobertura": g["cobertura"].mean(), "precisao": g["precisao"].mean(),
                   "area_metodo_vs_gabarito": g["area_metodo"].mean() / g["area_gabarito"].mean()})
resumo = pd.DataFrame(resumo).set_index("metodo")
salvar_tabela(resumo, SECAO, "dice-resumo")

fig, ax = plt.subplots(figsize=(8, 3.6))
for k, (nome, g) in enumerate(comparacao.groupby("metodo", sort=False)):
    ax.scatter(g["dice"], np.full(len(g), k) + rng.uniform(-0.12, 0.12, len(g)), color=CORES[nome], alpha=0.7, s=22)
    ax.errorbar(resumo.loc[nome, "dice_medio"], k, xerr=[[resumo.loc[nome, "dice_medio"] - resumo.loc[nome, "ic_baixo"]],
                [resumo.loc[nome, "ic_alto"] - resumo.loc[nome, "dice_medio"]]], fmt="D", color="#333", ms=6, capsize=4)
ax.set(yticks=range(len(resumo)), yticklabels=resumo.index, xlabel="Dice contra o contorno à mão", xlim=(0, 1),
       title="Concordância de área com o gabarito (média e IC 95 %)")
salvar_figura(fig, SECAO, "dice-metodos", comparacao)
plt.show()
resumo.round(3)

## 4. O ICT do contorno à mão

Com o contorno, a largura do coração sai direto: a maior extensão horizontal da máscara. Dividindo pela
largura do tórax medida pelo perfil, sai um **ICT com o coração recortado à mão**, que pode ser comparado
com o ICT automático do notebook 04 e com o rótulo.

In [ ]:
linhas = []
for _, r in marcacoes.iterrows():
    med = medidas[r["id"]]
    if not med.ok:
        continue
    larg = segmentacao.largura_maxima(segmentacao.poligono_para_mascara(r["pontos"]))
    linhas.append({"anotador": NOMES[r["anotador"]], "id": r["id"], "classe": classe[r["id"]],
                   "ict_automatico": med.ict, "ict_contorno": larg / med.torax_px})
icts = pd.DataFrame(linhas)
salvar_tabela(icts, SECAO, "ict-contorno", indice=False)
r_p, p_p = pearsonr(icts["ict_automatico"], icts["ict_contorno"])
r_s, _ = spearmanr(icts["ict_automatico"], icts["ict_contorno"])
aucs = {k: avaliacao.auc_bootstrap(icts["classe"].to_numpy(), icts[k].to_numpy(), n=2000) for k in ["ict_automatico", "ict_contorno"]}
print(f"Correlação ICT automático × ICT do contorno: Pearson {r_p:.2f} (p = {p_p:.3f}), Spearman {r_s:.2f}, n = {len(icts)}")
for k, a in aucs.items():
    print(f"AUC do {k.replace('_', ' ')} nestes {len(icts)} exames: {a['auc']:.2f} (IC 95 % {a['ic_baixo']:.2f}–{a['ic_alto']:.2f})")

fig, ax = plt.subplots(figsize=(5.5, 5))
for k in (0, 1):
    s = icts[icts["classe"] == k]
    ax.scatter(s["ict_automatico"], s["ict_contorno"], color=config.CORES_CLASSE[k], label=config.ROTULOS[k], s=40)
lim = [0.2, 0.9]
ax.plot(lim, lim, color="#999", ls="--", lw=1)
ax.axhline(0.5, color="#bbb", lw=0.8); ax.axvline(0.5, color="#bbb", lw=0.8)
ax.set(xlim=lim, ylim=lim, xlabel="ICT automático (perfil, notebook 04)", ylabel="ICT com o coração do contorno à mão",
       title=f"Dois jeitos de medir o ICT · r = {r_p:.2f}")
ax.legend()
salvar_figura(fig, SECAO, "ict-contorno", icts)
plt.show()

## 5. O Grad-CAM dentro do coração de verdade

No notebook 07 a "região do coração" era a caixa fixa. Com o contorno à mão dá para perguntar direto:
que fração da atenção da CNN cai **dentro do coração** daquele exame, comparada com a área dele?

In [ ]:
modelo = keras.models.load_model(config.MODELS / "cnn_pequena_crua.keras")
predicoes = pd.read_csv(config.DATA_PROCESSED / "predicoes.csv", sep=";").set_index("id")
limiar = avaliacao.limiar_youden(predicoes.loc[predicoes["conjunto"] == "validacao", "classe"].to_numpy(),
                                 predicoes.loc[predicoes["conjunto"] == "validacao", "CNN pequena · cru"].to_numpy())
ids_marcados = list(dele.index)
X = cnn.tensores(np.stack([imagens[i] for i in ids_marcados]))
previsto = (predicoes.loc[ids_marcados, "CNN pequena · cru"].to_numpy() >= limiar).astype(int)
mapas = gradcam.gradcam(modelo, X, classe=previsto)
linhas = []
for k, i in enumerate(ids_marcados):
    gab = segmentacao.poligono_para_mascara(dele[i])
    fr = float(gradcam.fracao_na_regiao(mapas[k : k + 1], gab)[0])
    linhas.append({"id": i, "classe": classe[i], "previsto": previsto[k], "fracao_no_coracao": fr,
                   "area_do_coracao": gab.mean(), "vezes_a_area": fr / gab.mean()})
atencao = pd.DataFrame(linhas)
salvar_tabela(atencao, SECAO, "gradcam-no-contorno", indice=False)
print(f"Atenção dentro do coração: {atencao['fracao_no_coracao'].median():.1%} (mediana), "
      f"para um coração que ocupa {atencao['area_do_coracao'].median():.1%} da imagem "
      f"→ {atencao['vezes_a_area'].median():.1f}× a área")
atencao.groupby(["classe", "previsto"])[["fracao_no_coracao", "vezes_a_area"]].median().rename(
    index=config.ROTULOS, level=0).round(2)

## 6. Concordância entre anotadores

Os exames 1 a 6 foram pensados para os três marcarem. Com mais de um anotador, o Dice **entre pessoas**
é o teto realista para um método automático: nenhum método precisa concordar com o gabarito mais do que
duas pessoas concordam entre si.

In [ ]:
if len(anotadores) < 2:
    print(f"Só {len(anotadores)} anotador marcou até agora: sem concordância entre pessoas para medir. "
          "Quando outros marcarem, exporte de novo a coleção `marcacoes` para data/processed/marcacoes/ e rode este notebook.")
else:
    pares = []
    tabela = marcacoes.pivot(index="id", columns="anotador", values="pontos")
    for i, linha in tabela.iterrows():
        quem = [a for a in anotadores if isinstance(linha.get(a), list)]
        for a in range(len(quem)):
            for b in range(a + 1, len(quem)):
                ma = segmentacao.poligono_para_mascara(linha[quem[a]])
                mb = segmentacao.poligono_para_mascara(linha[quem[b]])
                pares.append({"id": i, "par": f"{NOMES[quem[a]]} × {NOMES[quem[b]]}", "dice": dice(ma, mb)})
    pares = pd.DataFrame(pares)
    salvar_tabela(pares, SECAO, "concordancia-anotadores", indice=False)
    print(pares.groupby("par")["dice"].describe().round(3))

## Conclusões

* **O gabarito tem curva de aprendizado:** os exames 1 a 4 saíram com contornos pequenos (área mediana
  3,0 % da imagem) e os demais com 8,4 %. Os primeiros abraçam só o meio do mediastino. É um gabarito
  de **uma pessoa**, não radiologista, com 29 dos 30 exames: tudo abaixo herda essa limitação.
* **As bordas do ICT são o melhor recorte:** Dice médio **0,64** (IC 95 % 0,59–0,69), contra 0,58 do
  limiar e morfologia e 0,51 da caixa fixa. A área sai do tamanho certo (0,97 da área marcada);
  o que falta é o topo do coração, porque o perfil só mede as linhas 50 a 90 (cobertura 69 %).
* **Limiar e morfologia vaza:** cobre 89 % do coração marcado, mas com área 1,9 vez maior (precisão
  46 %). Diafragma e fígado são tão claros quanto o coração e se ligam a ele pela base; a abertura
  separa a coluna, não o abdome.
* **A caixa fixa cobre quase tudo e erra pela moldura:** 90 % de cobertura, 38 % de precisão. É a
  "moldura" da aula: pega o coração e muito em volta.
* **O ICT do contorno à mão separa pior que o automático aqui:** nestes 29 exames, AUC 0,61
  (IC 0,40–0,81) contra 0,87 (IC 0,73–0,98) do ICT automático, e os dois se correlacionam pouco
  (r = 0,38). Com n = 29 os intervalos se sobrepõem; e os contornos pequenos do começo puxam o ICT à
  mão para baixo. Não dá para concluir que a medida automática é melhor que a humana; dá para concluir
  que este gabarito é ruidoso.
* **A CNN olha o coração quando acerta a cardiomegalia:** nas cardiomegalias que ela detecta, a
  atenção dentro do coração marcado é **2,9 vezes** a área dele. Quando diz "normal", a atenção
  no coração fica abaixo da área (0,4 a 0,6×): ela procura em outro lugar, coerente com o notebook 07.

**Para melhorar o gabarito:** refazer os exames 1 a 4, marcar o 6, e ter Adriano e Leonardo marcando
os exames 1 a 6 em comum. A seção 6 mede então a concordância entre pessoas, que é o teto realista
para qualquer método automático. Basta exportar de novo a coleção `marcacoes` para
`data/processed/marcacoes/` e rodar o notebook.